In [1]:
import os
import sys
from importlib.machinery import SourceFileLoader
import torch
import torch.nn.functional as F
def find_root_path(path:str, word:str):
    parts = path.split(word, 1)    
    return parts[0] + word if len(parts) > 1 else path 

try:
    current_path = os.path.abspath(__file__)
except NameError:
    current_path = os.getcwd()
    
root_folder = find_root_path(os.path.abspath(current_path), 'art_lang')
sys.path.append(root_folder)

%load_ext autoreload
%autoreload 2

In [5]:

from rpod.decision_transformer.adapter import FrozenTextAdapter  
device =  "cuda" if torch.cuda.is_available() else "cpu"
print(device) 

MODEL = os.getenv("FTA_MODEL", "distilbert-base-uncased")   # this is encoder only 
# load adapter
adapter = FrozenTextAdapter(model_name=MODEL, out_dim=384, output_mode="tokens").to(device).eval()


cuda


In [6]:
# instructions 
texts = [
    "avoids the keep-out zone. Use a V-bar approach to dock at the +T port within 2 hours.",
    "Hold a standoff radius under 500 m before corridor entry; avoid the line-of-sight cone with half-angle 20 deg.",
    "No thrust during eclipse windows [600,900] s. Minimize fuel.",
    "Perform one fly-around before corridor entry. Minimize time.",
]

with torch.inference_mode():
    out = adapter(texts)  # forward pass 

print(out.size()) 

torch.Size([4, 30, 384])


### not used 

In [7]:
def _extract_pooled(adapter_output):
    """
    Accepts either:
      - Tensor: [B, D]
      - Object with .pooled (Tensor [B, D])
      - Dict with 'pooled' or ('hidden','attn_mask')
    Falls back to mean-pooling hidden states if available.
    """
    if torch.is_tensor(adapter_output):
        return adapter_output  # [B, D]
    # object with attribute
    if hasattr(adapter_output, "pooled") and torch.is_tensor(adapter_output.pooled):
        return adapter_output.pooled
    # dict with pooled
    if isinstance(adapter_output, dict) and torch.is_tensor(adapter_output.get("pooled", None)):
        return adapter_output["pooled"]
    # mean-pool tokens if hidden + attn_mask exist
    H = None
    M = None
    if hasattr(adapter_output, "hidden"):
        H = adapter_output.hidden
        M = getattr(adapter_output, "attn_mask", None)
    elif isinstance(adapter_output, dict):
        H = adapter_output.get("hidden", None)
        M = adapter_output.get("attn_mask", None)

    if torch.is_tensor(H):
        if M is None:
            # no mask: simple mean over tokens
            return H.mean(dim=1)
        else:
            # masked mean over content tokens
            M = M.float().unsqueeze(-1)  # [B,L,1]
            return (H * M).sum(dim=1) / (M.sum(dim=1).clamp_min(1e-8))

    raise ValueError("Could not find pooled/hidden outputs in adapter result.")

def cosine_matrix(X):
    # X: [N, D]
    Xn = F.normalize(X, dim=-1)
    return Xn @ Xn.T


In [8]:
# output printing
print("="*80)
print(f"Pooled embedding tensor shape: {tuple(z.shape)}")
norms = z.norm(dim=-1).cpu().numpy()
for i, (t, nrm) in enumerate(zip(texts, norms)):
    print(f"[{i}] ||z||={nrm:.4f}  |  {t[:80]}{'...' if len(t)>80 else ''}")

# Pairwise cosine similarities
S = cosine_matrix(z).cpu().numpy()
print("\nPairwise cosine similarity matrix (rows/cols correspond to the list order):")

with torch.no_grad():
    hdr = "      " + " ".join([f"{i:>6d}" for i in range(len(texts))])
    print(hdr)
    for i, row in enumerate(S):
        print(f"{i:>6d} " + " ".join([f"{v:6.3f}" for v in row]))


NameError: name 'z' is not defined